# Phase 6.0-B: 高亲和力调控网络分析

**项目**: Human LncRNA Atlas

**分析目标**: 识别具有强调控能力的核心 lncRNA 及其生物学特征

**数据源**: `/api/v1/export/high-affinity` API

---

## 分析流程

1. 数据获取与清洗
2. 描述性统计分析
3. Top 100 lncRNA 排行榜
4. 调控网络构建
5. 网络拓扑分析（中心性、社区检测）
6. 可视化与结果导出

## 1. 环境准备与数据获取

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy import stats
import requests
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

# 设置可视化样式
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# 设置中文字体（如果需要）
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print(f"分析开始时间: {datetime.now()}")
print(f"Pandas 版本: {pd.__version__}")
print(f"NetworkX 版本: {nx.__version__}")

In [ ]:
# API 配置
API_BASE_URL = "http://localhost:8000/api/v1"

# 数据获取参数
MIN_BA = 100  # 最小结合亲和力阈值
LIMIT = 10000  # 获取数量（可根据需要调整）
SPECIES_ID = None  # None = 所有物种，1=人类，2=黑猩猩，3=猕猴，4=狨猴

print(f"数据获取参数:")
print(f"  - 最小 BA: {MIN_BA}")
print(f"  - 数据量限制: {LIMIT}")
print(f"  - 物种筛选: {'所有物种' if SPECIES_ID is None else f'物种 {SPECIES_ID}'}")

In [ ]:
# 从 API 获取高亲和力调控关系数据
response = requests.get(
    f"{API_BASE_URL}/export/high-affinity",
    params={
        "min_ba": MIN_BA,
        "species_id": SPECIES_ID,
        "limit": LIMIT,
        "format": "json"
    }
)

if response.status_code == 200:
    data = response.json()
    df = pd.DataFrame(data['data'])
    print(f"✅ 成功获取 {len(df)} 条高亲和力调控关系")
    print(f"   查询参数: {data['query_params']}")
else:
    print(f"❌ API 请求失败: {response.status_code}")
    print(response.text)

# 显示数据前 5 行
df.head()

## 2. 数据清洗与预处理

In [ ]:
# 数据质量检查
print("=" * 60)
print("数据质量报告")
print("=" * 60)

print(f"\n数据维度: {df.shape}")
print(f"列名: {list(df.columns)}")

print(f"\n缺失值统计:")
print(df.isnull().sum())

print(f"\n数据类型:")
print(df.dtypes)

print(f"\n物种分布:")
print(df['species_name'].value_counts())

In [ ]:
# 数据清洗（如需要）
# 去除可能的重复记录
df_clean = df.drop_duplicates()
print(f"去重后数据量: {len(df_clean)} (移除了 {len(df) - len(df_clean)} 条重复记录)")

# 保存原始数据
df_clean.to_csv('data/high_affinity_raw_data.csv', index=False)
print("✅ 原始数据已保存到 data/high_affinity_raw_data.csv")

## 3. 描述性统计分析

In [ ]:
# 结合亲和力（BA）统计分析
print("=" * 60)
print("结合亲和力（Binding Affinity）统计分析")
print("=" * 60)

ba_stats = df_clean['binding_affinity'].describe()
print(ba_stats)

print(f"\n中位数: {df_clean['binding_affinity'].median():.2f}")
print(f"偏度: {stats.skew(df_clean['binding_affinity']):.2f}")
print(f"峰度: {stats.kurtosis(df_clean['binding_affinity']):.2f}")

In [ ]:
# BA 分布可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 直方图
axes[0, 0].hist(df_clean['binding_affinity'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Binding Affinity (BA)', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].set_title('BA Distribution (Histogram)', fontsize=14, fontweight='bold')
axes[0, 0].axvline(df_clean['binding_affinity'].mean(), color='red', 
                   linestyle='--', label=f'Mean = {df_clean["binding_affinity"].mean():.2f}')
axes[0, 0].legend()

# 2. 箱线图（按物种）
df_clean.boxplot(column='binding_affinity', by='species_name', ax=axes[0, 1])
axes[0, 1].set_xlabel('Species', fontsize=12)
axes[0, 1].set_ylabel('Binding Affinity', fontsize=12)
axes[0, 1].set_title('BA Distribution by Species', fontsize=14, fontweight='bold')
plt.sca(axes[0, 1])
plt.xticks(rotation=45)

# 3. 核密度估计
for species in df_clean['species_name'].unique():
    subset = df_clean[df_clean['species_name'] == species]['binding_affinity']
    axes[1, 0].hist(subset, bins=30, alpha=0.5, label=species, density=True)
axes[1, 0].set_xlabel('Binding Affinity', fontsize=12)
axes[1, 0].set_ylabel('Density', fontsize=12)
axes[1, 0].set_title('BA Density by Species', fontsize=14, fontweight='bold')
axes[1, 0].legend()

# 4. 累积分布函数（CDF）
for species in df_clean['species_name'].unique():
    subset = df_clean[df_clean['species_name'] == species]['binding_affinity'].sort_values()
    y = np.arange(1, len(subset) + 1) / len(subset)
    axes[1, 1].plot(subset, y, label=species, linewidth=2)
axes[1, 1].set_xlabel('Binding Affinity', fontsize=12)
axes[1, 1].set_ylabel('Cumulative Probability', fontsize=12)
axes[1, 1].set_title('Cumulative Distribution Function', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/01_ba_distribution_analysis.png', dpi=300, bbox_inches='tight')
print("✅ 图表已保存: figures/01_ba_distribution_analysis.png")
plt.show()

## 4. Top 100 lncRNA 排行榜

基于以下指标识别最重要的 lncRNA：
- 调控靶基因数量
- 平均结合亲和力
- 最大结合亲和力

In [ ]:
# 按 lncRNA 聚合统计
lncrna_stats = df_clean.groupby(['lncrna_gene_id', 'lncrna_name', 'species_name']).agg({
    'target_gene_id': 'count',  # 调控靶基因数量
    'binding_affinity': ['mean', 'max', 'std']  # BA 统计指标
}).reset_index()

# 重命名列
lncrna_stats.columns = [
    'lncrna_gene_id', 'lncrna_name', 'species_name',
    'target_count', 'avg_ba', 'max_ba', 'std_ba'
]

# 计算综合得分（调控数量 × 平均 BA）
lncrna_stats['综合得分'] = lncrna_stats['target_count'] * lncrna_stats['avg_ba']

# 排序获取 Top 100
top_100_lncrnas = lncrna_stats.nlargest(100, '综合得分')
top_100_lncrnas['排名'] = range(1, 101)

# 重新排列列顺序
top_100_lncrnas = top_100_lncrnas[[
    '排名', 'lncrna_name', 'species_name', 
    'target_count', 'avg_ba', 'max_ba', 'std_ba', '综合得分'
]]

print("=" * 80)
print("Top 100 高亲和力调控 lncRNA 排行榜")
print("=" * 80)
print(top_100_lncrnas.head(20))

# 导出到 Excel
top_100_lncrnas.to_excel('results/top_100_high_affinity_lncrnas.xlsx', index=False)
top_100_lncrnas.to_csv('results/top_100_high_affinity_lncrnas.csv', index=False)
print("\n✅ Top 100 排行榜已保存到 results/ 目录")

In [ ]:
# Top 20 lncRNA 可视化
top_20 = top_100_lncrnas.head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 左图：调控数量柱状图
axes[0].barh(top_20['lncrna_name'], top_20['target_count'], color='steelblue')
axes[0].set_xlabel('Number of Target Genes', fontsize=12)
axes[0].set_title('Top 20 lncRNAs by Regulation Count', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

# 右图：散点图（调控数量 vs 平均 BA）
scatter = axes[1].scatter(
    top_100_lncrnas['target_count'],
    top_100_lncrnas['avg_ba'],
    c=top_100_lncrnas['综合得分'],
    s=100,
    cmap='viridis',
    alpha=0.6,
    edgecolors='black'
)
axes[1].set_xlabel('Target Count', fontsize=12)
axes[1].set_ylabel('Average Binding Affinity', fontsize=12)
axes[1].set_title('lncRNA Characteristics Scatter Plot', fontsize=14, fontweight='bold')
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('Composite Score', fontsize=11)

# 标注 Top 5
for idx, row in top_100_lncrnas.head(5).iterrows():
    axes[1].annotate(
        row['lncrna_name'][:15],
        (row['target_count'], row['avg_ba']),
        fontsize=8,
        xytext=(5, 5),
        textcoords='offset points'
    )

plt.tight_layout()
plt.savefig('figures/02_top_lncrnas_visualization.png', dpi=300, bbox_inches='tight')
print("✅ 图表已保存: figures/02_top_lncrnas_visualization.png")
plt.show()

## 5. 调控网络构建

构建 lncRNA → 靶基因的有向调控网络，使用 NetworkX 进行拓扑分析。

In [ ]:
# 构建有向图（lncRNA → Target Gene）
G = nx.DiGraph()

# 添加节点和边
for _, row in df_clean.iterrows():
    # 添加 lncRNA 节点
    G.add_node(
        row['lncrna_name'],
        node_type='lncRNA',
        species=row['species_name'],
        gene_id=row['lncrna_gene_id']
    )
    
    # 添加靶基因节点
    G.add_node(
        row['target_name'],
        node_type='target_gene',
        species=row['species_name'],
        gene_id=row['target_gene_id']
    )
    
    # 添加调控边（权重 = BA）
    G.add_edge(
        row['lncrna_name'],
        row['target_name'],
        weight=row['binding_affinity'],
        chr=row['chr'],
        start=row['start_in_genome'],
        end=row['end_in_genome']
    )

print("=" * 60)
print("调控网络拓扑统计")
print("=" * 60)
print(f"总节点数: {G.number_of_nodes()}")
print(f"  - lncRNA 节点: {sum(1 for n, d in G.nodes(data=True) if d.get('node_type') == 'lncRNA')}")
print(f"  - 靶基因节点: {sum(1 for n, d in G.nodes(data=True) if d.get('node_type') == 'target_gene')}")
print(f"总边数: {G.number_of_edges()}")
print(f"网络密度: {nx.density(G):.6f}")
print(f"是否强连通: {nx.is_strongly_connected(G)}")
print(f"是否弱连通: {nx.is_weakly_connected(G)}")

## 6. 网络中心性分析

计算多种中心性指标，识别网络中最重要的节点。

In [ ]:
# 计算多种中心性指标
print("计算网络中心性指标...")

# 1. 度中心性（Degree Centrality）
degree_centrality = nx.degree_centrality(G)
in_degree_centrality = nx.in_degree_centrality(G)
out_degree_centrality = nx.out_degree_centrality(G)

# 2. 介数中心性（Betweenness Centrality）- 计算较慢
print("  计算介数中心性（可能需要几分钟）...")
betweenness_centrality = nx.betweenness_centrality(G, weight='weight', normalized=True)

# 3. 接近中心性（Closeness Centrality）
# 注意：对于不连通的图，计算每个连通分量
if nx.is_weakly_connected(G):
    closeness_centrality = nx.closeness_centrality(G, distance='weight')
else:
    print("  图不连通，跳过全局接近中心性计算")
    closeness_centrality = {}

print("✅ 中心性计算完成")

In [ ]:
# 汇总中心性结果（仅 lncRNA 节点）
centrality_df = pd.DataFrame([
    {
        'lncrna_name': node,
        'degree_centrality': degree_centrality.get(node, 0),
        'out_degree_centrality': out_degree_centrality.get(node, 0),
        'in_degree_centrality': in_degree_centrality.get(node, 0),
        'betweenness_centrality': betweenness_centrality.get(node, 0),
        'closeness_centrality': closeness_centrality.get(node, 0)
    }
    for node, data in G.nodes(data=True)
    if data.get('node_type') == 'lncRNA'
])

# 按介数中心性排序
centrality_df = centrality_df.sort_values('betweenness_centrality', ascending=False)

print("=" * 80)
print("Top 20 lncRNA - 介数中心性排行")
print("=" * 80)
print(centrality_df.head(20))

# 保存结果
centrality_df.to_excel('results/lncrna_centrality_analysis.xlsx', index=False)
print("\n✅ 中心性分析结果已保存: results/lncrna_centrality_analysis.xlsx")

In [ ]:
# 中心性分布可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 度中心性
axes[0, 0].hist(centrality_df['degree_centrality'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Degree Centrality', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Degree Centrality Distribution', fontsize=13, fontweight='bold')

# 出度中心性（lncRNA 调控能力）
axes[0, 1].hist(centrality_df['out_degree_centrality'], bins=30, 
                color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Out-Degree Centrality', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Regulatory Capacity Distribution', fontsize=13, fontweight='bold')

# 介数中心性
axes[1, 0].hist(centrality_df['betweenness_centrality'], bins=30, 
                color='lightgreen', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Betweenness Centrality', fontsize=11)
axes[1, 0].set_ylabel('Frequency', fontsize=11)
axes[1, 0].set_title('Betweenness Centrality Distribution', fontsize=13, fontweight='bold')

# Top 15 lncRNA 雷达图（多维度）
top_15 = centrality_df.head(15)
metrics = ['degree_centrality', 'out_degree_centrality', 'betweenness_centrality']
top_15_plot = top_15[['lncrna_name'] + metrics].set_index('lncrna_name')

# 归一化到 0-1
top_15_norm = (top_15_plot - top_15_plot.min()) / (top_15_plot.max() - top_15_plot.min())

# 使用热力图展示
sns.heatmap(top_15_norm.T, annot=True, fmt='.2f', cmap='YlOrRd', 
            ax=axes[1, 1], cbar_kws={'label': 'Normalized Score'})
axes[1, 1].set_xlabel('lncRNA', fontsize=11)
axes[1, 1].set_ylabel('Centrality Metric', fontsize=11)
axes[1, 1].set_title('Top 15 lncRNAs - Multi-Metric Heatmap', fontsize=13, fontweight='bold')
plt.setp(axes[1, 1].get_xticklabels(), rotation=90, fontsize=8)

plt.tight_layout()
plt.savefig('figures/03_centrality_analysis.png', dpi=300, bbox_inches='tight')
print("✅ 图表已保存: figures/03_centrality_analysis.png")
plt.show()

## 7. 社区检测分析

使用 Louvain 算法检测调控网络中的社区结构。

In [ ]:
# 转换为无向图进行社区检测
G_undirected = G.to_undirected()

# 使用 Louvain 算法进行社区检测
try:
    import community as community_louvain
    
    # 检测社区
    partition = community_louvain.best_partition(G_undirected, weight='weight')
    
    # 计算模块度
    modularity = community_louvain.modularity(partition, G_undirected, weight='weight')
    
    # 社区统计
    num_communities = len(set(partition.values()))
    
    print(f"社区检测结果:")
    print(f"  - 检测到 {num_communities} 个社区")
    print(f"  - 模块度 (Modularity): {modularity:.4f}")
    
    # 社区大小分布
    community_sizes = pd.Series(partition.values()).value_counts().sort_index()
    print(f"\n社区大小分布:")
    print(community_sizes.head(10))
    
except ImportError:
    print("⚠️ python-louvain 未安装，跳过社区检测")
    print("   安装命令: pip install python-louvain")
    partition = None

## 8. 网络可视化（子网络）

由于完整网络节点数量可能很大，我们选择 Top 50 lncRNA 及其靶基因进行可视化。

In [ ]:
# 提取 Top 30 lncRNA 的子网络
top_30_lncrnas = top_100_lncrnas.head(30)['lncrna_name'].tolist()

# 获取这些 lncRNA 及其所有靶基因
subgraph_nodes = set(top_30_lncrnas)
for lnc in top_30_lncrnas:
    if lnc in G:
        subgraph_nodes.update(G.successors(lnc))

# 创建子图
G_sub = G.subgraph(subgraph_nodes).copy()

print(f"子网络统计:")
print(f"  - 节点数: {G_sub.number_of_nodes()}")
print(f"  - 边数: {G_sub.number_of_edges()}")
print(f"  - 密度: {nx.density(G_sub):.6f}")

In [ ]:
# 网络可视化（使用 spring layout）
plt.figure(figsize=(20, 16))

# 计算布局
pos = nx.spring_layout(G_sub, k=2, iterations=50, seed=42)

# 节点颜色：lncRNA 为红色，靶基因为蓝色
node_colors = [
    'salmon' if G_sub.nodes[node].get('node_type') == 'lncRNA' else 'lightblue'
    for node in G_sub.nodes()
]

# 节点大小：根据度中心性
node_sizes = [
    300 + 3000 * degree_centrality.get(node, 0)
    for node in G_sub.nodes()
]

# 边的粗细：根据 BA
edge_widths = [
    0.5 + 2 * (G_sub[u][v]['weight'] / df_clean['binding_affinity'].max())
    for u, v in G_sub.edges()
]

# 绘制网络
nx.draw_networkx_nodes(G_sub, pos, node_color=node_colors, node_size=node_sizes, 
                       alpha=0.8, edgecolors='black', linewidths=1.5)
nx.draw_networkx_edges(G_sub, pos, width=edge_widths, alpha=0.4, 
                       edge_color='gray', arrows=True, arrowsize=15)
nx.draw_networkx_labels(G_sub, pos, font_size=7, font_weight='bold')

# 添加图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='salmon', edgecolor='black', label='lncRNA'),
    Patch(facecolor='lightblue', edgecolor='black', label='Target Gene')
]
plt.legend(handles=legend_elements, loc='upper right', fontsize=12)

plt.title('High Affinity Regulatory Network (Top 30 lncRNAs)', 
          fontsize=16, fontweight='bold', pad=20)
plt.axis('off')
plt.tight_layout()
plt.savefig('figures/04_regulatory_network_visualization.png', dpi=300, bbox_inches='tight')
print("✅ 网络图已保存: figures/04_regulatory_network_visualization.png")
plt.show()

## 9. 关键发现总结

In [ ]:
print("=" * 80)
print("高亲和力调控网络分析 - 关键发现")
print("=" * 80)

print(f"\n1. 数据规模")
print(f"   - 分析了 {len(df_clean)} 条高亲和力调控关系（BA > {MIN_BA}）")
print(f"   - 涉及 {len(centrality_df)} 个 lncRNA")
print(f"   - 调控 {sum(1 for n, d in G.nodes(data=True) if d.get('node_type') == 'target_gene')} 个靶基因")

print(f"\n2. 结合亲和力特征")
print(f"   - 平均 BA: {df_clean['binding_affinity'].mean():.2f}")
print(f"   - 中位数 BA: {df_clean['binding_affinity'].median():.2f}")
print(f"   - 最大 BA: {df_clean['binding_affinity'].max():.2f}")
print(f"   - BA > 200 的调控: {(df_clean['binding_affinity'] > 200).sum()} 条")

print(f"\n3. Top 1 lncRNA")
top_1 = top_100_lncrnas.iloc[0]
print(f"   - 名称: {top_1['lncrna_name']}")
print(f"   - 物种: {top_1['species_name']}")
print(f"   - 调控靶基因数: {top_1['target_count']}")
print(f"   - 平均 BA: {top_1['avg_ba']:.2f}")
print(f"   - 最大 BA: {top_1['max_ba']:.2f}")
print(f"   - 综合得分: {top_1['综合得分']:.2f}")

print(f"\n4. 网络拓扑特征")
print(f"   - 网络密度: {nx.density(G):.6f}")
if partition:
    print(f"   - 检测到 {num_communities} 个功能模块")
    print(f"   - 模块度: {modularity:.4f}")

print(f"\n5. 中心节点识别")
print(f"   - 最高介数中心性 lncRNA: {centrality_df.iloc[0]['lncrna_name']}")
print(f"     (介数中心性: {centrality_df.iloc[0]['betweenness_centrality']:.4f})")
print(f"   - 最高度中心性 lncRNA: {centrality_df.nlargest(1, 'degree_centrality').iloc[0]['lncrna_name']}")

print("\n" + "=" * 80)
print("分析完成！所有结果已保存到 results/ 和 figures/ 目录")
print("=" * 80)

## 10. 导出网络数据（用于 Cytoscape 等工具）

In [ ]:
# 导出节点表
nodes_df = pd.DataFrame([
    {
        'node_id': node,
        'node_type': data.get('node_type'),
        'species': data.get('species'),
        'degree_centrality': degree_centrality.get(node, 0),
        'betweenness_centrality': betweenness_centrality.get(node, 0)
    }
    for node, data in G.nodes(data=True)
])
nodes_df.to_csv('results/network_nodes.csv', index=False)

# 导出边表
edges_df = pd.DataFrame([
    {
        'source': u,
        'target': v,
        'binding_affinity': data['weight'],
        'chr': data.get('chr'),
        'start': data.get('start'),
        'end': data.get('end')
    }
    for u, v, data in G.edges(data=True)
])
edges_df.to_csv('results/network_edges.csv', index=False)

print("✅ 网络数据已导出:")
print("   - results/network_nodes.csv")
print("   - results/network_edges.csv")
print("   （可导入 Cytoscape 进行进一步可视化分析）")

---

## 分析总结

### 主要发现

1. **高亲和力调控关系特征**
   - 识别了 [数量] 条 BA > 100 的高质量调控关系
   - BA 分布呈 [描述分布特征]
   - 物种间 BA 分布存在 [差异/相似性]

2. **核心 lncRNA 识别**
   - Top 100 lncRNA 平均调控 [X] 个靶基因
   - 排名第一的 lncRNA 调控 [Y] 个靶基因
   - 综合得分显示 [发现]

3. **网络拓扑特征**
   - 网络呈现 [稠密/稀疏] 结构（密度 = [value]）
   - 检测到 [N] 个功能模块（模块度 = [value]）
   - 介数中心性分析揭示了 [关键节点]

### 生物学意义

- **调控能力强的 lncRNA** 可能在基因表达调控中起关键作用
- **高介数中心性节点** 可能是网络中的关键桥接节点
- **社区结构** 可能反映了不同的生物学功能模块

### 下一步分析方向

1. 对 Top lncRNA 的靶基因进行 **GO 功能富集分析**
2. 分析高 BA 调控位点的 **染色质状态特征**
3. 结合疾病数据，识别 **潜在治疗靶点**

---

**分析完成时间**: [运行时填充]

**下一个 Notebook**: `02_conservation_patterns.ipynb`